In [ ]:
# SQL Analysis Report for the Museum Comparison Project
#
# This notebook presents the SQL portion of the museum comparison project using the combined
# dataset created in the main EDA notebook. The analysis compares the Art Institute of Chicago
# and the Metropolitan Museum of Art across cultural representation, historical concentration,
# classification mix, and collection structure.



In [1]:
import pandas as pd
import sqlite3

# Load the combined dataset created in the EDA notebook
sql_df = pd.read_csv("combined_artworks.csv")

# Create SQLite in-memory database
conn = sqlite3.connect(':memory:')

# Convert likely numeric columns
for col in ['date_start', 'date_end', 'AccessionYear']:
    if col in sql_df.columns:
        sql_df[col] = pd.to_numeric(sql_df[col], errors='coerce')

# Push dataframe into SQLite
sql_df.to_sql('combined_artworks', conn, index=False, if_exists='replace')

# Optional check
pd.read_sql_query("SELECT * FROM combined_artworks LIMIT 5;", conn)


/tmp/ipykernel_688/4256672342.py:5: DtypeWarning: Columns (0: artist_display, 1: artwork_type_title, 2: date_display, 3: place_of_origin) have mixed types. Specify dtype option on import or set low_memory=False.
  sql_df = pd.read_csv("combined_artworks.csv")


,id,title,has_not_been_viewed_much,artist_title,artist_display,classification_title,artwork_type_title,department_title,medium_display,date_display,date_start,date_end,AccessionYear,place_of_origin,origin_normalized,century_num,century_label,museum
0,74026,"Garry Winogrand and Melissa, Los Angeles, Cali...",1,Tom Harney,"Tom Harney\nAmerican, born 1946",salted paper print,Photograph,Photography and Media,Gelatin silver print,1979,1979.0,1979.0,1989.0,United States,United States,20.0,20th century,AIC
1,215359,Setsugo (Joining Together)—A,1,Hamanishi Katsunori,"Hamanishi Katsunori\nJapanese, born 1949",print,Print,Arts of Asia,Mezzotint,1978,1978.0,1978.0,2013.0,Japan,Japan,20.0,20th century,AIC
2,132090,Bottle,1,Ancient Roman,Roman; Levant or Syria,bottle,Vessel,"Arts of Greece, Rome, and Byzantium","Glass, blown technique",2nd-3rd century,101.0,300.0,1945.0,Syria,Syria,2.0,Before 1000,AIC
3,186925,Arnold Toynbee,1,Yousuf Karsh,"Yousuf Karsh\nCanadian, born Turkish Armenia, ...",gelatin silver print,Photograph,Photography and Media,Gelatin silver print,1955,1955.0,1955.0,2014.0,Canada,Canada,20.0,20th century,AIC
4,146017,Sea Horse,1,Harold Eugene Edgerton,"Harold Eugene Edgerton\nAmerican, 1903–1990",gelatin silver (developing-out-paper) pr,Photograph,Photography and Media,Gelatin silver print,1939,1939.0,1939.0,1997.0,United States,United States,20.0,20th century,AIC


In [2]:
# Query 1 (Group By Query): Number of Works by Museum and Origin
#
# This query counts the number of artworks for each normalized origin category within each
# museum. The output answers: which cultural/geographic origins appear most often in each collection?
# The query starts from the combined table and filters out missing origin values so that the
# counts reflect only artworks that have a usable origin label. It then groups the data by both
# museum and origin_normalized. After grouping, COUNT(*) computes the total number of rows in
# each museum-origin combination. Finally, the results are ordered so the highest-count origins
# appear first within each museum. This query establishes the starting distribution. 
# It helps identify whether one museum is concentrated in a smaller set of origins
# or whether its holdings are spread more broadly across many origin categories.

query_1 = '''
-- Query 1 Label: Group By Query
SELECT
    museum,
    origin_normalized,
    COUNT(*) AS artwork_count
FROM combined_artworks
WHERE origin_normalized IS NOT NULL
GROUP BY museum, origin_normalized
ORDER BY museum, artwork_count DESC;
'''

result_1 = pd.read_sql_query(query_1, conn)
result_1.head(20)

,museum,origin_normalized,artwork_count
0,AIC,United States,403
1,AIC,France,160
2,AIC,Japan,132
3,AIC,United Kingdom,79
4,AIC,Italy,51
5,AIC,Germany,42
6,AIC,Netherlands,38
7,AIC,China,36
8,AIC,Egypt,13
9,AIC,Spain,12


In [3]:
# Query 2 (Group By Query): Number of Works by Museum and Time Period
#
# This query groups artworks into broad century-style time buckets using date_start and counts
# how many works each museum has in each period. It answers the question: how is each museum's
# collection distributed across historical time?
# The query first filters out rows with missing date_start values so the time calculations are
# valid. It then creates a derived field called time_period by dividing the year by 100,
# converting it to an integer, and multiplying by 100. This effectively creates century-like bins
# such as 1500, 1600, 1700, and so on. After that, it groups by museum and time_period and counts
# the number of artworks in each combination.
# This query shows whether the museums have stronger coverage in early periods, modern periods, 
# or specific historical eras. It helps explain whether observed representation patterns are driven 
# by time concentration.

query_2 = '''
-- Query 2 Label: Group By Query
SELECT
    museum,
    CAST((date_start / 100) AS INTEGER) * 100 AS time_period,
    COUNT(*) AS artwork_count
FROM combined_artworks
WHERE date_start IS NOT NULL
GROUP BY museum, time_period
ORDER BY museum, time_period;
'''

result_2 = pd.read_sql_query(query_2, conn)
result_2.head(20)

,museum,time_period,artwork_count
0,AIC,-2000,1
1,AIC,-1900,2
2,AIC,-1600,3
3,AIC,-1500,1
4,AIC,-1200,2
5,AIC,-1000,3
6,AIC,-800,1
7,AIC,-600,1
8,AIC,-500,1
9,AIC,-400,3


In [4]:
# Query 3 (Group By Query): Origin Representation Across Time Within Each Museum
#
# This query combines three variables at once: museum, origin, and time period. It counts how
# many artworks fall into each museum-origin-time combination and asks: how does
# cultural representation change across historical periods within each museum?
# The query filters to rows that have both a valid origin_normalized value and a valid
# date_start. It then creates the same century-style time_period bucket used in the prior query.
# Next, it groups by museum, origin_normalized, and time_period. The resulting count measures the
# number of artworks represented by a particular origin in a particular historical period inside
# a particular museum.
# Total origin counts can hide whether a culture is spread across many centuries or 
# concentrated in just one or two periods. This query helps distinguish between broad historical 
# representation and narrow historical concentration.

query_3 = '''
-- Query 3 Label: Group By Query
SELECT
    museum,
    origin_normalized,
    CAST((date_start / 100) AS INTEGER) * 100 AS time_period,
    COUNT(*) AS artwork_count
FROM combined_artworks
WHERE origin_normalized IS NOT NULL
  AND date_start IS NOT NULL
GROUP BY museum, origin_normalized, time_period
ORDER BY museum, origin_normalized, time_period;
'''

result_3 = pd.read_sql_query(query_3, conn)
result_3.head(20)

,museum,origin_normalized,time_period,artwork_count
0,AIC,Alexandria,200,1
1,AIC,Ancient Greece,-200,2
2,AIC,Argentina,1900,1
3,AIC,Askov,1800,1
4,AIC,Athens,-400,1
5,AIC,Augsburg,1600,1
6,AIC,Austria,1800,1
7,AIC,Austria,1900,1
8,AIC,Belgium,1500,2
9,AIC,Belgium,1600,6


In [5]:
# Query 4 (Group By Query): Origin and Classification by Museum
#
# This query counts artworks by museum, normalized origin, and classification. It answers the
# question: through what types of objects is each origin represented in each museum?
# The query first removes rows where either the origin or classification is missing, because both
# are necessary for the cross-tabulated count. It then groups by museum, origin_normalized, and
# classification_title. Each grouped result returns the number of artworks associated with that
# exact combination. Ordering the results by museum, origin, and descending count makes the most
# important classifications easier to identify within each origin category.
# A culture may be highly represented, but only through one narrow object type. Another culture 
# may appear across paintings, sculpture, decorative arts, textiles, and other forms. This query 
# helps measure that difference in breadth and gives more nuance to the representation discussion.

query_4 = '''
-- Query 4 Label: Group By Query
SELECT
    museum,
    origin_normalized,
    classification_title,
    COUNT(*) AS artwork_count
FROM combined_artworks
WHERE origin_normalized IS NOT NULL
  AND classification_title IS NOT NULL
GROUP BY museum, origin_normalized, classification_title
ORDER BY museum, origin_normalized, artwork_count DESC;
'''

result_4 = pd.read_sql_query(query_4, conn)
result_4.head(20)

,museum,origin_normalized,classification_title,artwork_count
0,AIC,Alexandria,coin,1
1,AIC,Ancient Greece,coin,2
2,AIC,Argentina,graphic design,1
3,AIC,Askov,textile,1
4,AIC,Athens,marble,1
5,AIC,Augsburg,case furniture,1
6,AIC,Austria,travel sketch,1
7,AIC,Austria,photograph,1
8,AIC,Belgium,engraving,4
9,AIC,Belgium,textile,3


In [6]:
# Query 5 (Group By Query): Classification Across Time by Museum
#
# This query counts artworks by museum, classification, and time period. Its purpose is to show
# how the mix of object types changes across history within each museum.
# The query filters for rows with usable classification and date values. It then computes the
# derived time_period bucket from date_start. After that, it groups by museum,
# classification_title, and time_period, and returns the count of artworks in each combination.
# This produces a structured summary of which classifications are most common in which historical
# eras.
# This query helps reveal whether a museum is strong in classifications during certain periods.
# For example, one museum may have strong early holdings in decorative arts while another may
# dominate later periods through paintings or prints. 

query_5 = '''
-- Query 5 Label: Group By Query
SELECT
    museum,
    classification_title,
    CAST((date_start / 100) AS INTEGER) * 100 AS time_period,
    COUNT(*) AS artwork_count
FROM combined_artworks
WHERE classification_title IS NOT NULL
  AND date_start IS NOT NULL
GROUP BY museum, classification_title, time_period
ORDER BY museum, classification_title, time_period;
'''

result_5 = pd.read_sql_query(query_5, conn)
result_5.head(20)

,museum,classification_title,time_period,artwork_count
0,AIC,Medu Art Ensemble,1900,1
1,AIC,albumen silver print,1800,7
2,AIC,albumen silver print,1900,1
3,AIC,amulet,-1000,1
4,AIC,aquatint,1700,1
5,AIC,aquatint,1800,1
6,AIC,aquatint,1900,4
7,AIC,architectural drawing,1900,12
8,AIC,architectural fragment,1800,1
9,AIC,architectural fragment,1900,1


In [12]:
# Query 6 (Group By Query): Side-by-Side Museum Comparison by Time Period

# This query places AIC and Met counts in the same row for each historical time bucket. Instead
# of returning separate rows by museum, it creates a direct side-by-side comparison across
# periods.
# The query groups the table only by derived time_period. Inside each group, it uses conditional
# aggregation with SUM(CASE WHEN ...) to count AIC rows separately from Met rows. This restructures 
# the output into a comparison table with one row per time period and one column for each museum's count.
# This output makes museum differences visible. Rather than comparing two separate grouped tables, the side-
# by-side layout shows where one institution has stronger historical depth than the other. This helps 
# identify periods where the museums diverge most clearly.

query_6 = '''
-- Query 6 Label: Group By Query
SELECT
    CAST((date_start / 100) AS INTEGER) * 100 AS time_period,
    SUM(CASE WHEN museum = 'AIC' THEN 1 ELSE 0 END) AS aic_count,
    SUM(CASE WHEN museum = 'Met' THEN 1 ELSE 0 END) AS met_count
FROM combined_artworks
WHERE date_start IS NOT NULL
GROUP BY time_period
ORDER BY time_period;
'''

result_6 = pd.read_sql_query(query_6, conn)
result_6.head(20)

,time_period,aic_count,met_count
0,-10000,0,1
1,-8000,0,6
2,-7500,0,14
3,-7200,0,19
4,-7000,0,1248
5,-6900,0,34
6,-6800,0,1
7,-6600,0,1
8,-6500,0,6
9,-6000,0,11


In [13]:
# Query 7 (Window Function Query): Running Total of Artworks Over Time
#
# This query calculates a cumulative running total of artworks over time for each museum. It
# answers the question: as history progresses, how quickly does each museum's represented
# collection build up?
# The query uses two layers; The inner query first groups by museum and time_period and counts
# the number of artworks in each period. The outer query then applies a window function:
# SUM(artwork_count) OVER (PARTITION BY museum ORDER BY time_period). The PARTITION BY museum
# clause resets the cumulative calculation for each museum, and the ORDER BY time_period clause
# ensures that the counts accumulate in order.
# A cumulative view reveals patterns that single-period counts do not. This query is useful for 
# showing the pace and breadth of historical accumulation.

query_7 = '''
-- Query 7 Label: Window Function Query
SELECT
    museum,
    time_period,
    artwork_count,
    SUM(artwork_count) OVER (
        PARTITION BY museum
        ORDER BY time_period
    ) AS running_total
FROM (
    SELECT
        museum,
        CAST((date_start / 100) AS INTEGER) * 100 AS time_period,
        COUNT(*) AS artwork_count
    FROM combined_artworks
    WHERE date_start IS NOT NULL
    GROUP BY museum, time_period
) t
ORDER BY museum, time_period;
'''

result_7 = pd.read_sql_query(query_7, conn)
result_7.head(20)

,museum,time_period,artwork_count,running_total
0,AIC,-2000,1,1
1,AIC,-1900,2,3
2,AIC,-1600,3,6
3,AIC,-1500,1,7
4,AIC,-1200,2,9
5,AIC,-1000,3,12
6,AIC,-800,1,13
7,AIC,-600,1,14
8,AIC,-500,1,15
9,AIC,-400,3,18


In [16]:
# Query 8 (Subquery Query): Origins Above the Museum Average
#
# This query identifies the origin categories whose counts are above the average origin count
# within the same museum. It highlights the most dominant origin groups relative to each museum's
# own internal distribution.
# The outer query groups the main table by museum and origin_normalized and counts artworks. The
# HAVING clause then filters those groups by comparing each count to a subquery-generated
# benchmark. The inner subquery calculates the average count across all origin groups inside the
# same museum. Only origins whose counts exceed that average are kept in the final result.
# This query is useful because it introduces a relative benchmark. Instead of asking only which
# origins have large counts, it asks which origins stand out compared with a typical origin inside 
# that museum; that makes the result more comparable across institutions.

query_8 = '''
-- Query 8 Label: Subquery Query
SELECT
    museum,
    origin_normalized,
    COUNT(*) AS artwork_count
FROM combined_artworks c
WHERE origin_normalized IS NOT NULL
GROUP BY museum, origin_normalized
HAVING COUNT(*) > (
    SELECT AVG(origin_count)
    FROM (
        SELECT COUNT(*) AS origin_count
        FROM combined_artworks c2
        WHERE c2.museum = c.museum
          AND c2.origin_normalized IS NOT NULL
        GROUP BY c2.origin_normalized
    )
)
ORDER BY museum, artwork_count DESC;
'''

result_8 = pd.read_sql_query(query_8, conn)
result_8.head(20)

,museum,origin_normalized,artwork_count
0,AIC,United States,403
1,AIC,France,160
2,AIC,Japan,132
3,AIC,United Kingdom,79
4,AIC,Italy,51
5,AIC,Germany,42
6,AIC,Netherlands,38
7,AIC,China,36
8,AIC,Egypt,13
9,AIC,Spain,12


In [17]:
# Query 9 (Subquery Query): Time Periods Above the Museum Average
#
# This query identifies the historical periods whose counts are above the average period count
# within each museum. It shows which eras are unusually strong relative to the rest of the
# museum's timeline.
# The outer query groups by museum and derived time_period, counting artworks in each period. The
# HAVING clause compares each period count to a correlated subquery. That subquery computes the
# average number of artworks across all time periods for the same museum. Only periods whose
# counts exceed that average are included in the output.
# This query is useful because it identifies the eras that are disproportionately important within 
# each institution. It helps pinpoint where each museum's strongest historical concentration lies

query_9 = '''
-- Query 9 Label: Subquery Query
SELECT
    museum,
    CAST((date_start / 100) AS INTEGER) * 100 AS time_period,
    COUNT(*) AS artwork_count
FROM combined_artworks c
WHERE date_start IS NOT NULL
GROUP BY museum, time_period
HAVING COUNT(*) > (
    SELECT AVG(period_count)
    FROM (
        SELECT COUNT(*) AS period_count
        FROM combined_artworks c2
        WHERE c2.museum = c.museum
          AND c2.date_start IS NOT NULL
        GROUP BY CAST((c2.date_start / 100) AS INTEGER) * 100
    )
)
ORDER BY museum, time_period;
'''

result_9 = pd.read_sql_query(query_9, conn)
result_9.head(20)

,museum,time_period,artwork_count
0,AIC,1500,46
1,AIC,1600,70
2,AIC,1700,144
3,AIC,1800,286
4,AIC,1900,554
5,Met,-600,6523
6,Met,-500,11712
7,Met,0,5210
8,Met,1400,5992
9,Met,1500,24077


In [19]:
# Query 10 (Subquery Query): Classifications Above the Museum Average
#
# This query identifies classifications whose counts are above the average classification count
# within each museum. It isolates the object types that are especially prominent relative to the
# museum's broader classification mix.
# The outer query groups rows by museum and classification_title and counts the number of
# artworks in each classification. The HAVING clause then compares those counts to a subquery
# that calculates the average classification count inside the same museum. The result is a
# filtered list of classifications that are stronger than a typical classification category
# within that institution.
# This query identifies the object categories that dominate each museum's holdings. 
# Used together with the prior two subqueries, it creates a framework across the project's 
# three main dimensions: origin, time, and classification.

query_10 = '''
-- Query 10 Label: Subquery Query
SELECT
    museum,
    classification_title,
    COUNT(*) AS artwork_count
FROM combined_artworks c
WHERE classification_title IS NOT NULL
GROUP BY museum, classification_title
HAVING COUNT(*) > (
    SELECT AVG(classification_count)
    FROM (
        SELECT COUNT(*) AS classification_count
        FROM combined_artworks c2
        WHERE c2.museum = c.museum
          AND c2.classification_title IS NOT NULL
        GROUP BY c2.classification_title
    )
)
ORDER BY museum, artwork_count DESC;
'''

result_10 = pd.read_sql_query(query_10, conn)
result_10.head(20)

,museum,classification_title,artwork_count
0,AIC,etching,124
1,AIC,gelatin silver (developing-out-paper) pr,104
2,AIC,lithograph,97
3,AIC,woodblock print,83
4,AIC,textile,83
5,AIC,photograph,66
6,AIC,photography,34
7,AIC,coin,29
8,AIC,print,28
9,AIC,pen and ink drawings,28


In [20]:
# Query 11 (Join Query): Joined Origin Comparison Between AIC and the Met
#
# This query creates a side-by-side origin comparison between the two museums using joins. Each
# row shows one origin category, the AIC count, the Met count, and the difference between the
# two.
# The query first builds two aggregated tables using common table expressions: one for AIC origin
# counts and one for Met origin counts. It then creates a combined list of all observed origins
# using a UNION. Finally, it joins that full origin list to the AIC and Met count tables with
# LEFT JOINs. COALESCE replaces missing values with zero so that origins present in one museum
# but not the other are still retained in the comparison.
# This query shows where the two museums differ. This helps identify over or underrepresentation 
# by institution and provides evidence for museum-level differences in cultural coverage.

query_11 = '''
-- Query 11 Label: Join Query
WITH aic_counts AS (
    SELECT
        origin_normalized,
        COUNT(*) AS aic_count
    FROM combined_artworks
    WHERE museum = 'AIC'
      AND origin_normalized IS NOT NULL
    GROUP BY origin_normalized
),
met_counts AS (
    SELECT
        origin_normalized,
        COUNT(*) AS met_count
    FROM combined_artworks
    WHERE museum = 'Met'
      AND origin_normalized IS NOT NULL
    GROUP BY origin_normalized
),
all_origins AS (
    SELECT origin_normalized FROM aic_counts
    UNION
    SELECT origin_normalized FROM met_counts
)
SELECT
    ao.origin_normalized,
    COALESCE(a.aic_count, 0) AS aic_count,
    COALESCE(m.met_count, 0) AS met_count,
    COALESCE(a.aic_count, 0) - COALESCE(m.met_count, 0) AS count_difference
FROM all_origins ao
LEFT JOIN aic_counts a
    ON ao.origin_normalized = a.origin_normalized
LEFT JOIN met_counts m
    ON ao.origin_normalized = m.origin_normalized
ORDER BY ABS(COALESCE(a.aic_count, 0) - COALESCE(m.met_count, 0)) DESC,
         ao.origin_normalized;
'''

result_11 = pd.read_sql_query(query_11, conn)
result_11.head(20)

,origin_normalized,aic_count,met_count,count_difference
0,United States,403,80120,-79717
1,France,160,38890,-38730
2,United Kingdom,79,22883,-22804
3,Italy,51,20500,-20449
4,Germany,42,10159,-10117
5,Netherlands,38,8135,-8097
6,Japan,132,6038,-5906
7,Spain,12,2168,-2156
8,China,36,1869,-1833
9,Austria,2,1820,-1818


In [21]:
# Query 12 (Join Query): Join Origin Counts to Average Artwork Year by Museum and Origin
#
# This query joins two aggregated summaries at the museum-origin level: the number of artworks
# and the average start year. It answers the question: for each origin, how large is the group
# and how historically early or late is it on average?
# The first common table expression counts artworks by museum and origin_normalized. The second
# calculates the average date_start for the same museum-origin pairs. These two summaries are
# then joined on both museum and origin_normalized, which are the shared keys. Because both
# tables operate at the same level of aggregation, the join is meaningful and avoids duplication.
# The result is a compact table that combines size and temporal center into one output.
# This join is useful because it connects representation volume with historical placement. 
# An origin may have a high artwork count, but those artworks may be concentrated in a
# relatively narrow or modern time window. Another origin may have fewer works but a much earlier
# historical footprint. This query helps discuss scale and temporal character together.

query_12 = '''
-- Query 12 Label: Join Query
WITH origin_counts AS (
    SELECT
        museum,
        origin_normalized,
        COUNT(*) AS artwork_count
    FROM combined_artworks
    WHERE origin_normalized IS NOT NULL
    GROUP BY museum, origin_normalized
),
origin_avg_year AS (
    SELECT
        museum,
        origin_normalized,
        AVG(date_start) AS avg_start_year
    FROM combined_artworks
    WHERE origin_normalized IS NOT NULL
      AND date_start IS NOT NULL
    GROUP BY museum, origin_normalized
)
SELECT
    c.museum,
    c.origin_normalized,
    c.artwork_count,
    y.avg_start_year
FROM origin_counts c
JOIN origin_avg_year y
    ON c.museum = y.museum
   AND c.origin_normalized = y.origin_normalized
ORDER BY c.museum, c.artwork_count DESC, y.avg_start_year;
'''

result_12 = pd.read_sql_query(query_12, conn)
result_12.head(20)

,museum,origin_normalized,artwork_count,avg_start_year
0,AIC,United States,403,1949.090226
1,AIC,France,160,1852.543750
2,AIC,Japan,132,1837.863636
3,AIC,United Kingdom,79,1844.468354
4,AIC,Italy,51,1532.647059
5,AIC,Germany,42,1792.690476
6,AIC,Netherlands,38,1662.631579
7,AIC,China,36,1176.242424
8,AIC,Egypt,13,-1067.153846
9,AIC,Belgium,12,1702.250000


In [22]:
# Query 13 (Join Query): Join Classification Counts to Average Accession Year by Museum and
# Classification
#
# This query joins classification-level counts to classification-level average accession year. It
# asks: which object types are most common in each museum, and how recently were those types
# typically accessioned?
# The first CTE groups by museum and classification_title to count artworks. The second CTE
# groups by the same keys and computes the average AccessionYear, after excluding missing values.
# The final JOIN combines the two summary tables on both museum and classification. This
# structure keeps the join precise because it matches the same level of aggregation on both
# sides.
# This query links classification prominence with accession timing. In the broader project 
# context, it helps explain whether dominant classifications are longstanding strengths of 
# the collection or whether they reflect more recent acquisition emphasis. That can support
# a better context about how the museums built their present-day profile.

query_13 = '''
-- Query 13 Label: Join Query
WITH classification_counts AS (
    SELECT
        museum,
        classification_title,
        COUNT(*) AS artwork_count
    FROM combined_artworks
    WHERE classification_title IS NOT NULL
    GROUP BY museum, classification_title
),
classification_accession AS (
    SELECT
        museum,
        classification_title,
        AVG(AccessionYear) AS avg_accession_year
    FROM combined_artworks
    WHERE classification_title IS NOT NULL
      AND AccessionYear IS NOT NULL
    GROUP BY museum, classification_title
)
SELECT
    c.museum,
    c.classification_title,
    c.artwork_count,
    a.avg_accession_year
FROM classification_counts c
JOIN classification_accession a
    ON c.museum = a.museum
   AND c.classification_title = a.classification_title
ORDER BY c.museum, c.artwork_count DESC, a.avg_accession_year;
'''

result_13 = pd.read_sql_query(query_13, conn)
result_13.head(20)

,museum,classification_title,artwork_count,avg_accession_year
0,AIC,etching,124,1960.233766
1,AIC,gelatin silver (developing-out-paper) pr,104,1988.271186
2,AIC,lithograph,97,1962.590909
3,AIC,woodblock print,83,1950.388889
4,AIC,textile,83,1965.088889
5,AIC,photograph,66,2007.500000
6,AIC,photography,34,2009.000000
7,AIC,coin,29,1973.041667
8,AIC,pen and ink drawings,28,1947.894737
9,AIC,print,28,2010.157895


In [23]:
# Query 14 (Window Function Query): Ranked Origin Representation Within Each Museum

# This query ranks origin categories within each museum by artwork count. Instead of only listing
# counts, it creates an ordered hierarchy showing which origins are first, second, third, and so
# on within each institution.
# The query begins with a CTE that counts artworks by museum and origin_normalized. It then
# applies the DENSE_RANK() window function. The PARTITION BY museum clause tells the database to
# restart the ranking separately for each museum, and the ORDER BY artwork_count DESC clause
# ranks origins from largest to smallest. DENSE_RANK() is used instead of ROW_NUMBER() so tied
# counts receive the same rank rather than being forced into separate positions.
# Ranking is especially useful when many categories are present and raw counts become harder to
# interpret quickly. This query makes it easier to discuss the most important origin groups in 
# each museum and compare representation across institutions.

query_14 = '''
-- Query 14 Label: Window Function Query
WITH origin_counts AS (
    SELECT
        museum,
        origin_normalized,
        COUNT(*) AS artwork_count
    FROM combined_artworks
    WHERE origin_normalized IS NOT NULL
    GROUP BY museum, origin_normalized
)
SELECT
    museum,
    origin_normalized,
    artwork_count,
    DENSE_RANK() OVER (
        PARTITION BY museum
        ORDER BY artwork_count DESC
    ) AS origin_rank
FROM origin_counts
ORDER BY museum, origin_rank, origin_normalized;
'''

result_14 = pd.read_sql_query(query_14, conn)
result_14.head(20)

,museum,origin_normalized,artwork_count,origin_rank
0,AIC,United States,403,1
1,AIC,France,160,2
2,AIC,Japan,132,3
3,AIC,United Kingdom,79,4
4,AIC,Italy,51,5
5,AIC,Germany,42,6
6,AIC,Netherlands,38,7
7,AIC,China,36,8
8,AIC,Egypt,13,9
9,AIC,Belgium,12,10
